In [0]:
dbutils.widgets.text('silver_table',"unique_carriers")
dbutils.widgets.text('checkpoint_path',"unique_carriers")
dbutils.widgets.text('primary_key','Code')
dbutils.widgets.text('file_format','parquet')
dbutils.widgets.text('source_file_name','UNIQUE_CARRIERS')

In [0]:
from datetime import datetime
from pyspark.sql.functions import *

In [0]:
# dbutils.setwidget("target_table", "projectdev.cleansed.airlines")
# dbutils.setwidget("checkpoint_path", "/Volumes/projectdev/bronze/airlines/checkpoint/")

In [0]:
from delta.tables import DeltaTable
catalog = 'projectdev'
schema = 'cleansed'
file_format = dbutils.widgets.get('file_format')
target_table = f"{catalog}.{schema}.{dbutils.widgets.get('silver_table')}"

# source_path = f'abfss://raw@datalakestrgdev.dfs.core.windows.net/{dbutils.widgets.get('source_file_name')}/{datetime.now().strftime('year=%Y/month=%m/day=%d')}/'

source_path = (
    f"abfss://raw@datalakestrgdev.dfs.core.windows.net/"
    f"{dbutils.widgets.get('source_file_name')}/"
)

checkpoint_base = f'/Volumes/projectdev/bronze/{dbutils.widgets.get('checkpoint_path')}'
checkpoint_path = f"{checkpoint_base}/checkpoint/"

primary_key = dbutils.widgets.get('primary_key')

def upsert_to_delta(batch_df, batchId):
    # spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
    try:
        batch_df = (
                batch_df
                .filter(col(primary_key).isNotNull())
                .dropDuplicates([primary_key])
            )
        if not spark.catalog.tableExists(target_table):
            batch_df.write.format("delta").mode("overwrite").saveAsTable(target_table)
        else:
            delta_table = DeltaTable.forName(spark, target_table)

            delta_table.alias("tgt").merge(
                batch_df.alias("src"),
                f"tgt.{primary_key} = src.{primary_key}"
            ).whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
    except Exception as e:
        print(f" Batch {batchId} failed")
        print(str(e))
        raise


(
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", f"{file_format}")
        .option("cloudFiles.schemaLocation", f"{checkpoint_base}/schema")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.inferColumnTypes", True)
        .load(source_path)
        .writeStream
        .foreachBatch(upsert_to_delta)
        .option('mergeSchema','true')
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .start()
)

